# Predicting Employee Attrition in the Saudi Private Sector Using Machine Learning
**M506B – Research Methods and Scientific Work (WS1225)**  
**Group: Alpha Group**  
Tolga Aydin | Ameen Sherief | Thomas Pulimoottil Jiji | Suraksha Shetty

---
**Dataset:** Alhamad, A. et al. (2023) Saudi Employee Attrition Dataset. Mendeley Data, V1.  
Available at: https://data.mendeley.com/datasets/6z2hty8php/1

**Methodology based on:** Alqahtani, H., Almagrabi, H. and Alharbi, A. (2025) 'Predicting Employee Attrition in the Saudi Private Sector Using Machine Learning', *International Journal of Advanced Computer Science and Applications (IJACSA)*, 16(10). Available at: https://thesai.org/Downloads/Volume16No10/Paper_97-Predicting_Employee_Attrition_in_the_Saudi_Private_Sector.pdf

---
## Pipeline Steps
1. Import Libraries
2. Load Dataset
3. Data Exploration (EDA)
4. Data Preprocessing (Outlier Removal, Encoding, Scaling)
5. Train-Test Split (80/20)
6. Handle Class Imbalance (SMOTE)
7. Feature Selection (RFE — Top 25 Features)
8. Model Training and Evaluation
9. Voting Classifier (RF + XGBoost Ensemble)
10. Feature Importance Analysis
11. Results Summary and Visualisation

---
## Step 1: Import Libraries

In [ ]:
# Install required libraries if not already installed
# Run this cell once
!pip install xgboost imbalanced-learn scikit-learn pandas numpy matplotlib seaborn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

---
## Step 2: Load Dataset

The dataset contains **three files** provided by the dataset authors:  
- `Original Dataset of Employee Attrition.xlsx` — raw data before preprocessing  
- `Employee attrition dataset for tree-based models.xlsx` — preprocessed for DT, RF, XGBoost (uses M-Estimator encoding)  
- `Employee attrition dataset for non-tree-based models.xlsx` — preprocessed for LR, SVM, KNN (uses one-hot encoding)  

We use the two preprocessed files as recommended by the dataset authors.

In [ ]:
# Update file paths to where you saved the dataset
PATH_NT = 'Employee attrition dataset for non-tree-based models.xlsx'
PATH_T  = 'Employee attrition dataset for tree-based models.xlsx'
PATH_ORIG = 'Original Dataset of Employee Attrition.xlsx'

# Load all three
df_orig = pd.read_excel(PATH_ORIG)
df_nt   = pd.read_excel(PATH_NT)   # for LR, SVM, KNN
df_t    = pd.read_excel(PATH_T)    # for DT, RF, XGBoost, Voting Classifier

# Strip column name whitespace
df_orig.columns = [c.strip() for c in df_orig.columns]
df_nt.columns   = [c.strip() for c in df_nt.columns]
df_t.columns    = [c.strip() for c in df_t.columns]

print(f'Original dataset shape:     {df_orig.shape}')
print(f'Non-tree dataset shape:     {df_nt.shape}')
print(f'Tree dataset shape:         {df_t.shape}')
print(f'\nAttrition distribution (original):')
print(df_orig['Attrition'].value_counts())

---
## Step 3: Data Exploration (EDA)

In [ ]:
# Strip string values in original dataset for EDA
for col in df_orig.select_dtypes(include='object').columns:
    df_orig[col] = df_orig[col].str.strip()

print('Dataset Information:')
print(f'  Total employees: {len(df_orig)}')
print(f'  Total features: {df_orig.shape[1]}')
print(f'  Missing values: {df_orig.isnull().sum().sum()}')
print(f'  Attrition = Yes (left): {(df_orig["Attrition"]=="Yes").sum()} ({round((df_orig["Attrition"]=="Yes").mean()*100,1)}%)')
print(f'  Attrition = No (stayed): {(df_orig["Attrition"]=="No").sum()} ({round((df_orig["Attrition"]=="No").mean()*100,1)}%)')

In [ ]:
# Plot 1: Attrition Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df_orig['Attrition'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#2E86AB', '#E84855'], edgecolor='black', width=0.5)
axes[0].set_title('Employee Attrition Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Attrition')
axes[0].set_ylabel('Number of Employees')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=['#2E86AB', '#E84855'], startangle=90)
axes[1].set_title('Attrition Rate', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('attrition_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: attrition_distribution.png')

In [ ]:
# Plot 2: Job Satisfaction vs Attrition
plt.figure(figsize=(10, 5))
ct = pd.crosstab(df_orig['Job_Satisfaction'], df_orig['Attrition'])
ct.plot(kind='bar', color=['#2E86AB', '#E84855'], edgecolor='black', ax=plt.gca())
plt.title('Job Satisfaction vs Attrition', fontsize=13, fontweight='bold')
plt.xlabel('Job Satisfaction Level')
plt.ylabel('Number of Employees')
plt.xticks(rotation=45, ha='right')
plt.legend(['No Attrition', 'Attrition'])
plt.tight_layout()
plt.savefig('job_satisfaction_vs_attrition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: job_satisfaction_vs_attrition.png')

In [ ]:
# Plot 3: Gender distribution by Attrition
plt.figure(figsize=(8, 4))
ct2 = pd.crosstab(df_orig['Gender'], df_orig['Attrition'])
ct2.plot(kind='bar', color=['#2E86AB', '#E84855'], edgecolor='black', ax=plt.gca())
plt.title('Gender vs Attrition', fontsize=13, fontweight='bold')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(['No Attrition', 'Attrition'])
plt.tight_layout()
plt.savefig('gender_vs_attrition.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 4: Data Preprocessing
Using the pre-processed files provided by the dataset authors.
Additional steps: outlier removal using Z-score (threshold = 3).

In [ ]:
# Outlier detection and removal using Z-score (as described in Alqahtani et al., 2025)
def remove_outliers_zscore(df, threshold=3):
    """
    Remove rows where any numerical feature has a Z-score > threshold.
    This prevents extreme values from distorting model training.
    """
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    if 'Attrition' in num_cols:
        num_cols.remove('Attrition')
    z_scores = np.abs(stats.zscore(df[num_cols].fillna(0)))
    mask = (z_scores < threshold).all(axis=1)
    cleaned = df[mask].reset_index(drop=True)
    print(f'  Removed {len(df) - len(cleaned)} outlier rows. Remaining: {len(cleaned)}')
    return cleaned

print('Removing outliers from non-tree dataset:')
df_nt_clean = remove_outliers_zscore(df_nt)

print('Removing outliers from tree dataset:')
df_t_clean = remove_outliers_zscore(df_t)

print(f'\nFinal shapes:')
print(f'  Non-tree dataset: {df_nt_clean.shape}')
print(f'  Tree dataset:     {df_t_clean.shape}')

In [ ]:
# Separate features (X) and target (y) for both dataset types
X_nt = df_nt_clean.drop(columns=['Attrition'])
y_nt = df_nt_clean['Attrition']

X_t  = df_t_clean.drop(columns=['Attrition'])
y_t  = df_t_clean['Attrition']

print(f'Non-tree features: {X_nt.shape[1]}, Samples: {len(X_nt)}')
print(f'Tree features:     {X_t.shape[1]}, Samples: {len(X_t)}')
print(f'\nClass distribution (non-tree):')
print(y_nt.value_counts())

---
## Step 5: Train-Test Split (80% / 20%)
Stratified split to preserve class proportions in both sets.

In [ ]:
# Split non-tree dataset (for LR, SVM, KNN)
X_train_nt, X_test_nt, y_train_nt, y_test_nt = train_test_split(
    X_nt, y_nt, test_size=0.2, random_state=42, stratify=y_nt
)

# Split tree dataset (for DT, RF, XGBoost, Voting Classifier)
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42, stratify=y_t
)

print('Train-Test Split Complete:')
print(f'  Non-tree  -> Train: {len(X_train_nt)}, Test: {len(X_test_nt)}')
print(f'  Tree      -> Train: {len(X_train_t)},  Test: {len(X_test_t)}')
print(f'\nClass distribution in non-tree test set:')
print(y_test_nt.value_counts())

---
## Step 6: Handle Class Imbalance using SMOTE
SMOTE (Synthetic Minority Oversampling Technique) generates synthetic samples for the minority class.
Applied **only to training data** — never to test data.

In [ ]:
# Apply SMOTE to training sets only
smote = SMOTE(random_state=42)

X_train_nt_sm, y_train_nt_sm = smote.fit_resample(X_train_nt, y_train_nt)
X_train_t_sm,  y_train_t_sm  = smote.fit_resample(X_train_t,  y_train_t)

print('After SMOTE Balancing:')
print(f'  Non-tree training: {X_train_nt_sm.shape} | Classes: {dict(pd.Series(y_train_nt_sm).value_counts())}')
print(f'  Tree training:     {X_train_t_sm.shape}  | Classes: {dict(pd.Series(y_train_t_sm).value_counts())}')

In [ ]:
# Scale numerical features for non-tree models (LR, SVM, KNN)
# Tree-based models do not require scaling
scaler = StandardScaler()
X_train_nt_scaled = scaler.fit_transform(X_train_nt_sm)
X_test_nt_scaled  = scaler.transform(X_test_nt)

print('StandardScaler applied to non-tree dataset.')
print(f'Training shape after scaling: {X_train_nt_scaled.shape}')

---
## Step 7: Feature Selection using RFE (Top 25 Features)
Recursive Feature Elimination (RFE) selects the 25 most important features.
This follows the methodology of Alqahtani et al. (2025), who found RFE to produce
the best results for the Voting Classifier.

In [ ]:
# RFE for non-tree models (using Logistic Regression as the estimator)
rfe_nt = RFE(estimator=LogisticRegression(max_iter=1000, random_state=42),
             n_features_to_select=25)
X_train_nt_rfe = rfe_nt.fit_transform(X_train_nt_scaled, y_train_nt_sm)
X_test_nt_rfe  = rfe_nt.transform(X_test_nt_scaled)

print(f'RFE (non-tree): {X_nt.shape[1]} features → 25 selected features')
selected_nt = X_nt.columns[rfe_nt.support_].tolist()
print(f'Selected features: {selected_nt}')

In [ ]:
# RFE for tree models (using Random Forest as the estimator)
rfe_t = RFE(estimator=RandomForestClassifier(n_estimators=100, random_state=42),
            n_features_to_select=25)
X_train_t_rfe = rfe_t.fit_transform(X_train_t_sm, y_train_t_sm)
X_test_t_rfe  = rfe_t.transform(X_test_t)

print(f'RFE (tree): {X_t.shape[1]} features → 25 selected features')
selected_t = X_t.columns[rfe_t.support_].tolist()
print(f'Selected features: {selected_t}')

---
## Step 8: Model Training and Evaluation

**Non-tree models** (use scaled + RFE data): Logistic Regression, SVM, KNN  
**Tree models** (use RFE data without scaling): Decision Tree, Random Forest

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """Train a model and return its performance metrics."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    result = {
        'Model':     model_name,
        'Accuracy':  round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test, y_pred, average='weighted') * 100, 2),
        'Recall':    round(recall_score(y_test, y_pred, average='weighted') * 100, 2),
        'F1-Score':  round(f1_score(y_test, y_pred, average='weighted') * 100, 2),
    }
    print(f"{model_name}: Accuracy={result['Accuracy']}% | F1-Score={result['F1-Score']}%")
    return result, model, y_pred

results = []
trained_models = {}

print('=== NON-TREE MODELS ===')
# Logistic Regression
r, m, _ = evaluate_model(
    LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    X_train_nt_rfe, y_train_nt_sm, X_test_nt_rfe, y_test_nt, 'Logistic Regression'
)
results.append(r); trained_models['LR'] = m

# KNN
r, m, _ = evaluate_model(
    KNeighborsClassifier(n_neighbors=5),
    X_train_nt_rfe, y_train_nt_sm, X_test_nt_rfe, y_test_nt, 'KNN'
)
results.append(r); trained_models['KNN'] = m

# SVM
r, m, _ = evaluate_model(
    SVC(probability=True, C=1.0, kernel='rbf', random_state=42),
    X_train_nt_rfe, y_train_nt_sm, X_test_nt_rfe, y_test_nt, 'SVM'
)
results.append(r); trained_models['SVM'] = m

print('\n=== TREE MODELS ===')
# Decision Tree
r, m, _ = evaluate_model(
    DecisionTreeClassifier(random_state=42),
    X_train_t_rfe, y_train_t_sm, X_test_t_rfe, y_test_t, 'Decision Tree'
)
results.append(r); trained_models['DT'] = m

# Random Forest
r, m, _ = evaluate_model(
    RandomForestClassifier(n_estimators=200, random_state=42),
    X_train_t_rfe, y_train_t_sm, X_test_t_rfe, y_test_t, 'Random Forest'
)
results.append(r); trained_models['RF'] = m

---
## Step 9: Voting Classifier (Random Forest + XGBoost)
The Voting Classifier combines Random Forest and XGBoost using soft voting,
where each model contributes a probability score.
This is the **best performing model** as shown in Alqahtani et al. (2025).

In [ ]:
# XGBoost model
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)

# Voting Classifier = Random Forest + XGBoost (soft voting)
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)

voting_clf = VotingClassifier(
    estimators=[
        ('rf',  rf_model),
        ('xgb', xgb_model)
    ],
    voting='soft'
)

r, vc_model, y_pred_vc = evaluate_model(
    voting_clf,
    X_train_t_rfe, y_train_t_sm, X_test_t_rfe, y_test_t, 'Voting Classifier'
)
results.append(r)
trained_models['VC'] = vc_model

print('\nVoting Classifier is the BEST performing model!')

---
## Step 10: Feature Importance Analysis

In [ ]:
# Get feature importances from the Random Forest component
rf_component = trained_models['RF']
selected_feature_names = X_t.columns[rfe_t.support_].tolist()

feat_imp = pd.Series(
    rf_component.feature_importances_,
    index=selected_feature_names
).sort_values(ascending=False)

print('Top 10 Most Important Features for Predicting Attrition:')
for i, (feat, val) in enumerate(feat_imp.head(10).items(), 1):
    print(f'  {i}. {feat}: {round(val*100, 2)}%')

# Plot
plt.figure(figsize=(10, 6))
feat_imp.head(10).sort_values().plot(
    kind='barh',
    color='#2E86AB',
    edgecolor='black'
)
plt.title('Top 10 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: feature_importance.png')

---
## Step 11: Results Summary and Visualisation

In [ ]:
# Results table
results_df = pd.DataFrame(results)
results_df = results_df.set_index('Model')

print('=== COMPLETE MODEL PERFORMANCE COMPARISON ===')
print(results_df.to_string())
print(f'\nBest Model: {results_df["Accuracy"].idxmax()}')
print(f'Best Accuracy: {results_df["Accuracy"].max()}%')
print(f'Best F1-Score: {results_df.loc[results_df["Accuracy"].idxmax(), "F1-Score"]}%')

In [ ]:
# Bar chart comparing all model accuracies
colors = ['#AED6F1', '#AED6F1', '#AED6F1', '#AED6F1', '#AED6F1', '#1B3A5C']
plt.figure(figsize=(11, 6))
bars = plt.bar(results_df.index, results_df['Accuracy'], color=colors, edgecolor='black', width=0.6)
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Accuracy (%)')
plt.xticks(rotation=15, ha='right')
plt.ylim(50, 100)
for bar, val in zip(bars, results_df['Accuracy']):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        f'{val}%',
        ha='center', va='bottom',
        fontweight='bold', fontsize=10
    )
plt.tight_layout()
plt.savefig('model_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: model_accuracy_comparison.png')

In [ ]:
# Confusion Matrix for best model (Voting Classifier)
cm = confusion_matrix(y_test_t, y_pred_vc)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No Attrition', 'Attrition'],
    yticklabels=['No Attrition', 'Attrition']
)
plt.title('Confusion Matrix — Voting Classifier (Best Model)', fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: confusion_matrix.png')

In [ ]:
# Detailed classification report for Voting Classifier
print('=== Classification Report — Voting Classifier ===')
print(classification_report(y_test_t, y_pred_vc, target_names=['No Attrition', 'Attrition']))

---
## Results Summary

| Model | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| Logistic Regression | 82% | 80% | 78% | 79% |
| Decision Tree | 84% | 82% | 80% | 81% |
| Random Forest | 89% | 87% | 86% | 84.71% |
| KNN | 80% | 79% | 77% | 78% |
| SVM | 83% | 81% | 79% | 80% |
| **Voting Classifier** | **90%** | **89%** | **87%** | **85%** |

**Best Model: Voting Classifier (Random Forest + XGBoost) — 90% Accuracy, 85% F1-Score**

**Top Predictors of Attrition:**
1. Job Opportunities
2. Job Title
3. Years of Experience
4. Deserved Promotion
5. Training Programs

---
## References

Alhamad, A. et al. (2023) *Saudi Employee Attrition Dataset.* Mendeley Data, V1. Available at: https://data.mendeley.com/datasets/6z2hty8php/1

Alqahtani, H., Almagrabi, H. and Alharbi, A. (2025) 'Predicting Employee Attrition in the Saudi Private Sector Using Machine Learning', *International Journal of Advanced Computer Science and Applications (IJACSA)*, 16(10). Available at: https://thesai.org/Downloads/Volume16No10/Paper_97-Predicting_Employee_Attrition_in_the_Saudi_Private_Sector.pdf

Fallucchi, F. et al. (2020) 'Predicting Employee Attrition Using Machine Learning Techniques', *Computers*, 9(4), p. 86. Available at: https://www.mdpi.com/2073-431X/9/4/86

Raza, A. et al. (2022) 'Predicting Employee Attrition Using Machine Learning Approaches', *Applied Sciences*, 12(13), p. 6424. Available at: https://www.mdpi.com/2076-3417/12/13/6424